In [20]:
import pandas as pd
import numpy as np
import sqlite3
df = pd.read_csv('adult.csv')

print("Shape:", df.shape)
df.head()

Shape: (32561, 15)


,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [21]:
#Task 1

print("Missing values BEFORE fix (misleading):")
print(df.isnull().sum())

print("\nUnique workclass values (note the hidden '?' marker):")
print(df['workclass'].unique())

df.replace('?', np.nan, inplace=True)
df.replace(' ?', np.nan, inplace=True)

print("\nMissing values AFTER fix:")
print(df.isnull().sum())

Missing values BEFORE fix (misleading):
age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

Unique workclass values (note the hidden '?' marker):
['?' 'Private' 'State-gov' 'Federal-gov' 'Self-emp-not-inc' 'Self-emp-inc'
 'Local-gov' 'Without-pay' 'Never-worked']

Missing values AFTER fix:
age                  0
workclass         1836
fnlwgt               0
education            0
education.num        0
marital.status       0
occupation        1843
relationship         0
race                 0
sex                  0
capital.gain         0
capital.loss         0
hours.per.week       0
native.country     583
income               0
dtype: int64


In [22]:
#Task 2
missing_count = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct
}).sort_values('missing_count', ascending=False)

print(missing_summary[missing_summary['missing_count'] > 0])
print("\nColumns with the most missing data: occupation, workclass, native.country")

                missing_count  missing_pct
occupation               1843         5.66
workclass                1836         5.64
native.country            583         1.79

Columns with the most missing data: occupation, workclass, native.country


In [23]:
#Task 3
df['workclass'] = df['workclass'].fillna('Unknown')
df['occupation'] = df['occupation'].fillna('Unknown')
df['native.country'] = df['native.country'].fillna(df['native.country'].mode()[0])
 
print("Total missing values remaining:", df.isnull().sum().sum())


# Decisions:
# - workclass & occupation (categorical): impute with an explicit Unknown category —
#   dropping would lose 5.6% of rows, and mode-imputation would falsely assume missing
#   respondents match the majority class.
# - native.country (categorical): impute with the mode (United-States) — this column is
#   heavily dominated by one category (90%) and missing only 1.8%, so mode imputation
#   introduces little distortion.
# - No numeric column has missing values, so no numeric drop/impute
#   decision is needed.

Total missing values remaining: 0


In [24]:
#Task 4
exact_dupes = df.duplicated().sum()
print("Exact duplicate rows:", exact_dupes)

cols_without_target = [c for c in df.columns if c != 'income']
dupes_ignoring_income = df.duplicated(subset=cols_without_target).sum()
print("Duplicate rows ignoring:", dupes_ignoring_income)

df.drop_duplicates(inplace=True)
print("Shape after dropping exact duplicates:", df.shape)

# Decision: There is no unique ID column, so two different people can genuinely share
# every attribute. Exact full-row duplicates (income included) are dropped as true
# duplicates; rows that only match when `income` is ignored are kept, since they likely
# represent distinct individuals who happen to share every other attribute.

Exact duplicate rows: 24
Duplicate rows ignoring: 25
Shape after dropping exact duplicates: (32537, 15)


In [27]:
#Task 5
for col in ['education', 'marital.status', 'native.country']:
    print(f" {col} ({df[col].nunique()} unique values)")
    print(sorted(df[col].dropna().unique()))
    has_whitespace = df[col].dropna().apply(lambda x: x != x.strip()).sum()
    print(f"Values with leading/trailing whitespace: {has_whitespace}\n")

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

 education (16 unique values)
['10th', '11th', '12th', '1st-4th', '5th-6th', '7th-8th', '9th', 'Assoc-acdm', 'Assoc-voc', 'Bachelors', 'Doctorate', 'HS-grad', 'Masters', 'Preschool', 'Prof-school', 'Some-college']
Values with leading/trailing whitespace: 0

 marital.status (7 unique values)
['Divorced', 'Married-AF-spouse', 'Married-civ-spouse', 'Married-spouse-absent', 'Never-married', 'Separated', 'Widowed']
Values with leading/trailing whitespace: 0

 native.country (41 unique values)
['Cambodia', 'Canada', 'China', 'Columbia', 'Cuba', 'Dominican-Republic', 'Ecuador', 'El-Salvador', 'England', 'France', 'Germany', 'Greece', 'Guatemala', 'Haiti', 'Holand-Netherlands', 'Honduras', 'Hong', 'Hungary', 'India', 'Iran', 'Ireland', 'Italy', 'Jamaica', 'Japan', 'Laos', 'Mexico', 'Nicaragua', 'Outlying-US(Guam-USVI-etc)', 'Peru', 'Philippines', 'Poland', 'Portugal', 'Puerto-Rico', 'Scotland', 'South', 'Taiwan', 'Thailand', 'Trinadad&Tobago', 'United-States', 'Vietnam', 'Yugoslavia']
Values w

In [30]:
#Task 6
print(df.describe())
print()
print(df.describe(include='object'))

age_mean, age_median = df['age'].mean(), df['age'].median()
hours_mean, hours_median = df['hours.per.week'].mean(), df['hours.per.week'].median()

print(f"\nAge    mean: {age_mean:.2f}, median: {age_median:.1f}")
print(f"Hours  mean: {hours_mean:.2f}, median: {hours_median:.1f}")
print("\nBoth columns show mean approximately equals median (age is very mildly right-skewed by a smaller")
print("group of older workers; hours-per-week clusters tightly around the standard 40).")

                age        fnlwgt  education.num  capital.gain  capital.loss  \
count  32537.000000  3.253700e+04   32537.000000  32537.000000  32537.000000   
mean      38.585549  1.897808e+05      10.081815   1078.443741     87.368227   
std       13.637984  1.055565e+05       2.571633   7387.957424    403.101833   
min       17.000000  1.228500e+04       1.000000      0.000000      0.000000   
25%       28.000000  1.178270e+05       9.000000      0.000000      0.000000   
50%       37.000000  1.783560e+05      10.000000      0.000000      0.000000   
75%       48.000000  2.369930e+05      12.000000      0.000000      0.000000   
max       90.000000  1.484705e+06      16.000000  99999.000000   4356.000000   

       hours.per.week  
count    32537.000000  
mean        40.440329  
std         12.346889  
min          1.000000  
25%         40.000000  
50%         40.000000  
75%         45.000000  
max         99.000000  

       workclass education      marital.status      occupation

In [32]:
#Task 7
print("(a) Occupation counts:")
print(df['occupation'].value_counts().head(5))
print("Most common occupation:", df['occupation'].value_counts().idxmax())

print("\n(b) Sex distribution (%):")
print((df['sex'].value_counts(normalize=True) * 100).round(2))

print("\n(c) Income distribution (%):")
print((df['income'].value_counts(normalize=True) * 100).round(2))

(a) Occupation counts:
occupation
Prof-specialty     4136
Craft-repair       4094
Exec-managerial    4065
Adm-clerical       3768
Sales              3650
Name: count, dtype: int64
Most common occupation: Prof-specialty

(b) Sex distribution (%):
sex
Male      66.92
Female    33.08
Name: proportion, dtype: float64

(c) Income distribution (%):
income
<=50K    75.91
>50K     24.09
Name: proportion, dtype: float64


In [33]:
#Task 8
edu_check = df.groupby('education')['education.num'].unique()
print(edu_check)

inconsistent = edu_check[edu_check.apply(len) > 1]
print(f"\nEducation levels with inconsistent education.num values: {len(inconsistent)}")
if len(inconsistent) > 0:
    print(inconsistent)
else:
    print("No inconsistencies found — education and education.num are perfectly 1-to-1.")

education
10th             [6]
11th             [7]
12th             [8]
1st-4th          [2]
5th-6th          [3]
7th-8th          [4]
9th              [5]
Assoc-acdm      [12]
Assoc-voc       [11]
Bachelors       [13]
Doctorate       [16]
HS-grad          [9]
Masters         [14]
Preschool        [1]
Prof-school     [15]
Some-college    [10]
Name: education.num, dtype: object

Education levels with inconsistent education.num values: 0
No inconsistencies found — education and education.num are perfectly 1-to-1.


In [34]:
#Task 9
conn = sqlite3.connect(':memory:')
df.to_sql('adult_income', conn, index=False, if_exists='replace')

query = "SELECT * FROM adult_income WHERE age > 30;"
df_sql = pd.read_sql_query(query, conn)

manual_filter_count = (df['age'] > 30).sum()

print("SQL-extracted DataFrame shape (age > 30):", df_sql.shape)
print("Rows with age > 30 (manual pandas filter):", manual_filter_count)
print("Match:", manual_filter_count == df_sql.shape[0])

conn.close()

SQL-extracted DataFrame shape (age > 30): (21979, 15)
Rows with age > 30 (manual pandas filter): 21979
Match: True


In [ ]:
# Task 10

# Dataset Overview
The Census Income dataset: 32,561 rows, 15 columns of 1994 US
Census demographic/employment data. Each row is one individual; the target income is
binary (<=50K vs >50K).

# Data Quality Issues Found
- Hidden missing values: encoded as "?" instead of NaN. After fixing:
  occupation (1,843, 5.66%), `workclass` (1,836, 5.64%), `native.country` (583, 1.79%).
  No numeric column had missing values.
- **Duplicates:** 24 exact full-row duplicates dropped; 25 rows matched only when `income`
  was ignored and were kept (no ID column, so attribute matches ≠ same person).
- **Inconsistent categorical data:** no whitespace inconsistencies found in this CSV
  export; `.str.strip()` applied defensively anyway.

### Key Observations
- Income is **imbalanced** (~76% `<=50K` vs ~24% `>50K`, ~3:1).
- Occupation is fairly even; `Prof-specialty` is the most common (~13.5%).
- Sex is imbalanced in-sample (~67% Male / ~33% Female).
- Age (mean 38.6 / median 37) and hours-per-week (mean 40.4 / median 40) show no meaningful
  skew.
- `education` and `education.num` are perfectly consistent (1-to-1) — redundant encodings.

### Cleaning Decisions Made
1. Replaced `"?"` with `NaN` across the dataset.
2. Imputed `workclass`/`occupation` with `"Unknown"`; imputed `native.country` with its
   mode.
3. Dropped 24 exact duplicate rows; kept income-only mismatches.
4. Stripped whitespace on all text columns defensively.
5. Verified SQL extraction matched a manual pandas filter exactly.